<a href="https://colab.research.google.com/github/prakashgoud421/FragmentTransaction/blob/fragment/privategpt_colab_tToSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install gradio transformers sqlalchemy


In [20]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
from sqlalchemy import create_engine, inspect

db_url = "sqlite:////content/drive/MyDrive/models/testdb.sqlite"

def extract_schema(db_url):
    engine = create_engine(db_url)
    inspector = inspect(engine)
    schema = []
    for table in inspector.get_table_names():
        columns = inspector.get_columns(table)
        col_names = [col['name'] for col in columns]
        schema.append(f"{table}({', '.join(col_names)})")
    return "\n".join(schema)


In [14]:
import json
import re
from sqlalchemy import create_engine, inspect
import gradio as gr
from ctransformers import AutoModelForCausalLM

# Load model from Drive
model = AutoModelForCausalLM.from_pretrained(
    '/content/drive/MyDrive/models/',
    model_file='mistral-7b-instruct-v0.1.Q4_K_M.gguf',
    model_type='mistral',
    gpu_layers=20  # Optional: tweak this for performance
)

# SQLite DB
db_url = "sqlite:////content/drive/MyDrive/models/testdb.sqlite"  # 4 slashes for absolute path

def extract_schema(db_url):
    engine = create_engine(db_url)
    inspector = inspect(engine)
    schema = {}
    for table in inspector.get_table_names():
        columns = inspector.get_columns(table)
        schema[table] = [col['name'] for col in columns]
    return json.dumps(schema)

def to_sql_query(user_query):
    schema = extract_schema(db_url)
    prompt = f"""
    You are a SQL generator. Given the following schema and a user question, output only the SQL statement.

    Schema: {schema}
    User question: {user_query}
    Output (SQL only):
    """
    output = model(prompt)
    sql = re.split(r'Output \(SQL only\):', output)[-1].strip()
    return sql

# Launch Gradio app
gr.Interface(
    fn=to_sql_query,
    inputs=gr.Textbox(label="Enter your question"),
    outputs=gr.Textbox(label="Generated SQL"),
    title="Text-to-SQL Converter",
    description="Convert natural language questions into SQL queries using Mistral model."
).launch(share=True)  # share=True to get public link


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://118237da5214b8bcf9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
import json
import re
from sqlalchemy import create_engine, inspect
import gradio as gr
from ctransformers import AutoModelForCausalLM

# Load model from Drive
model = AutoModelForCausalLM.from_pretrained(
    '/content/drive/MyDrive/models/',
    model_file='mistral-7b-instruct-v0.1.Q4_K_M.gguf',
    model_type='mistral',
    gpu_layers=20  # Optional: tweak for your GPU
)

# SQLite DB
db_url = "sqlite:////content/drive/MyDrive/models/testdb.sqlite"  # absolute path

def extract_schema(db_url):
    engine = create_engine(db_url)
    inspector = inspect(engine)
    schema = {}
    for table in inspector.get_table_names():
        columns = inspector.get_columns(table)
        schema[table] = [col['name'] for col in columns]
    return json.dumps(schema, indent=2)

def to_sql_query(user_query):
    schema = extract_schema(db_url)

    # Few-shot prompt examples for better SQL generation
    few_shot_examples = """
### Example 1
User question: List all users in the "users" table.
SQL: SELECT * FROM users;

### Example 2
User question: Get the names of all employees who joined after 2020.
SQL: SELECT name FROM employees WHERE join_date > '2020-01-01';

### Example 3
User question: Count the number of orders in the "orders" table.
SQL: SELECT COUNT(*) FROM orders;
"""

    prompt = f"""
You are a SQL generator that converts natural language questions into SQL queries.

Schema: {schema}

Instructions:
- Output only the SQL query.
- Do NOT include any explanations or additional text.
- Use proper SQL syntax according to the schema.
- If the question is ambiguous or missing info, do your best based on schema.

Here are some examples:
{few_shot_examples}

User question: {user_query}
SQL:
"""

    output = model(prompt)
    # Extract SQL part from model output
    sql = output.strip().split('SQL:')[-1].strip()
    return sql

# Launch Gradio app
gr.Interface(
    fn=to_sql_query,
    inputs=gr.Textbox(label="Enter your question"),
    outputs=gr.Textbox(label="Generated SQL"),
    title="Text-to-SQL Converter",
    description="Convert natural language questions into SQL queries using Mistral model."
).launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://883180513342ccd272.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
